# CryptoGraph — run the experiments on Colab (T4 GPU)

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU***.
2. In Google Drive create the folder **`MyDrive/CryptoGraph/`** and upload the two files made on the laptop:
   - `cryptograph_code.zip` (source code + configs, ~0.3 MB)
   - `cryptograph_data_v2.zip` (processed hourly data with a SHA-256 manifest, ~250 MB)
3. *Runtime → Run all.* Allow Drive access when asked.

Every finished run is copied to `MyDrive/CryptoGraph/colab_runs/` immediately, so a disconnect loses at most the run in
progress (re-running the notebook skips runs that are already on Drive). At the end, **send back
`MyDrive/CryptoGraph/cryptograph_colab_results.zip`**: it holds every run (config, environment, log, metrics, the saved
model `checkpoint.pt`, per-event predictions) and the summary table.

Data and code stay in your private Drive and this Colab session; nothing is uploaded anywhere else.

In [ ]:
# 1. Settings — the only cell you might edit
DRIVE_DIR = "/content/drive/MyDrive/CryptoGraph"
CODE_ZIP = f"{DRIVE_DIR}/cryptograph_code.zip"
DATA_ZIP = f"{DRIVE_DIR}/cryptograph_data_v2.zip"
WORK = "/content/CryptoGraph"

SEEDS = [0, 1, 2, 3, 4]
EXPERIMENTS = [                                   # (config, seeds) — run in this order
    ("configs/experiment/lstm_1h_shock_final.yaml", SEEDS),   # T2: next-hour shock, LSTM (no graph)
    ("configs/experiment/lstm_1h_rv_final.yaml", SEEDS),      # T3: realized volatility, LSTM (no graph)
    ("configs/experiment/tgn_1h_final_v2.yaml", [0]),         # GPU replication of one TGN seed (compare with CPU run)
]
DEVICE = "cuda"            # "cuda" on a GPU runtime; "cpu" also works
RUN_TESTS = True

In [ ]:
# 2. Mount Drive, check the GPU and the input files
import os, subprocess, sys, json, shutil, time
from google.colab import drive
drive.mount("/content/drive")
for f in (CODE_ZIP, DATA_ZIP):
    assert os.path.exists(f), f"missing {f}: upload it to {DRIVE_DIR}"
    print(f"found {os.path.basename(f)} ({os.path.getsize(f) / 1e6:.1f} MB)")
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout
      or "no GPU visible: set Runtime → Change runtime type → T4 GPU")

In [ ]:
# 3. Unpack the code and install (Colab's CUDA build of torch already satisfies torch>=2.0)
import zipfile
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(WORK)
    code_manifest = json.loads(z.read("PACKAGE_MANIFEST.json"))
os.chdir(WORK)
# no .git on Colab: runs record the commit the code package was made from
os.environ["CRYPTOGRAPH_GIT_COMMIT"] = code_manifest["git"]["commit"]
os.environ["CRYPTOGRAPH_GIT_DIRTY"] = str(code_manifest["git"]["dirty"]).lower()
print(f"code from commit {code_manifest['git']['commit']}" + (" (with uncommitted changes)" if code_manifest["git"]["dirty"] else ""))
!pip install -q torch_geometric
!pip install -q -e . --no-deps
# verify the code against its own SHA-256 manifest (fails loudly on any difference)
!python -m cryptograph.pipelines.package_data unpack "{CODE_ZIP}" --dest "{WORK}"
import torch
print("torch", torch.__version__, "| CUDA", torch.version.cuda, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
# 4. Unpack the data (every file's SHA-256 is checked against the manifest)
t0 = time.time()
!python -m cryptograph.pipelines.package_data unpack "{DATA_ZIP}" --dest "{WORK}"
print(f"data ready in {time.time() - t0:.0f}s")

In [ ]:
# 5. Unit tests on this machine (same suite as on the laptop)
if RUN_TESTS:
    !python -m pytest -q -p no:cacheprovider tests/unit/training tests/unit/analysis tests/unit/network 2>&1 | tail -3

In [ ]:
# 6. Speed check: one TGN training pass over 100 batches on CPU vs GPU (decides where TGN runs should go)
timing = subprocess.run([sys.executable, "-W", "ignore", "-c", r"""
import time, torch, numpy as np
from cryptograph.training.tgn_task import StreamRunner, node_feature_table
from cryptograph.models.tgn import ContagionTGN
from cryptograph.training.losses import focal_loss
base = 'data/processed/v2/1h/'
p = torch.load(base + 'panel.pt', weights_only=False); ev = torch.load(base + 'tgn_events.pt', weights_only=False)
net = torch.load(base + 'network_features.pt', weights_only=False)
d = ev['full']; feats = node_feature_table(p, net, ev['graph_index'])
w = {s: p['sample_mask'][s].numpy() for s in ('train', 'val', 'test')}
for dev in (['cpu', 'cuda'] if torch.cuda.is_available() else ['cpu']):
    torch.manual_seed(0)
    m = ContagionTGN(30, 5, feats.shape[-1]).to(dev)
    r = StreamRunner(m, d, feats, w, torch.device(dev), 200); r.reset()
    opt = torch.optim.Adam(m.parameters(), 5e-4)
    t0 = time.time(); r.run(0, len(d.t), 'train', opt, lambda a, b, c: focal_loss(a, b, c), max_batches=100)
    if dev == 'cuda': torch.cuda.synchronize()
    print(f'{dev}: {(time.time() - t0) / 100 * 1000:.1f} ms per training batch')
"""], capture_output=True, text=True)
print(timing.stdout or timing.stderr[-2000:])

In [ ]:
# 7. Train. Each finished run is copied to Drive at once; runs already on Drive are skipped.
SAVE = f"{DRIVE_DIR}/colab_runs"
os.makedirs(SAVE, exist_ok=True)

def done_on_drive(cfg, seed):
    """True if a finished run of this experiment and seed is already on Drive (run dir = time_exp_seed_commit)."""
    exp = [l.split(":", 1)[1].strip() for l in open(cfg) if l.startswith("name:")][0]
    for d in os.listdir(SAVE):
        parts = d[16:].rsplit("_", 2)
        if len(parts) == 3 and parts[0] == exp and parts[1] == str(seed) and os.path.exists(f"{SAVE}/{d}/metrics.json"):
            return True, exp
    return False, exp

log = []
for cfg, seeds in EXPERIMENTS:
    for seed in seeds:
        done, exp = done_on_drive(cfg, seed)
        if done:
            print(f"skip {exp} seed {seed}: already on Drive"); continue
        t0 = time.time()
        r = subprocess.run([sys.executable, "-W", "ignore", "-m", "cryptograph.pipelines.train", "--config", cfg,
                            "--seed", str(seed), "--device", DEVICE], capture_output=True, text=True)
        out = r.stdout.strip().splitlines()
        run_dir = next((l.split("run dir:", 1)[1].strip() for l in out if l.startswith("run dir:")), None)
        status = "OK" if r.returncode == 0 and run_dir else "FAILED"
        if run_dir:
            shutil.copytree(run_dir, f"{SAVE}/{os.path.basename(run_dir)}", dirs_exist_ok=True)
        msg = f"{status} {exp} seed {seed} ({(time.time() - t0) / 60:.1f} min)"
        print(msg + ("" if status == "OK" else "\n" + r.stderr[-1500:]))
        log.append(msg)
        if os.path.exists("runs/results.csv"):
            shutil.copy("runs/results.csv", f"{SAVE}/results_colab.csv")

In [ ]:
# 8. Summary: mean ± std over seeds (validation and test) for every experiment run here
import numpy as np, pandas as pd
rows = []
for d in sorted(os.listdir(SAVE)):
    f = f"{SAVE}/{d}/metrics.json"
    if not os.path.exists(f):
        continue
    m = json.load(open(f)); m.pop("history", None)
    exp, seed, _ = d[16:].rsplit("_", 2)
    rows.append({"experiment": exp, "seed": int(seed), **{k: v for k, v in m.items() if isinstance(v, (int, float))}})
res = pd.DataFrame(rows)
cols = [c for c in res.columns if c.startswith(("val_", "test_")) and ("auprc" in c or "mse" in c or "auroc" in c)]
summary = res.groupby("experiment")[cols].agg(["mean", "std"]).round(4)
pd.set_option("display.max_columns", 200)
display(res.groupby("experiment")["seed"].apply(sorted))
display(summary.T)
summary.to_csv(f"{SAVE}/summary_colab.csv")

In [ ]:
# 9. Zip everything for sending back: all runs (models, predictions, metrics, logs) + summary
out = f"{DRIVE_DIR}/cryptograph_colab_results"
shutil.make_archive(out, "zip", SAVE)
print(f"Send back: {out}.zip ({os.path.getsize(out + '.zip') / 1e6:.1f} MB)")